# DAY 13 -- Persistence (Files & Contexts)

#### Files & Context Managers

##### File modes (optional parameter for `open`)


* `'w'`: create/overwrite (wipe)
* `'a'`: create/append (keep old content)
* `'x'`: create but fail if exists (safe create)
* `'r'`: read [default]
* Add `'b'` for bytes: `'rb'`, `'wb'`, `'ab'`

##### Context manager (what `with` guarantees)

```
enter resource    --->    do work    --->    exit resource
     __enter__                               __exit__  (runs even on exceptions)
```


##### Buffering (why disk doesn’t spin per line)

```
your writes  -->  Python buffer  -->  OS buffer  -->  disk
                      (RAM)             (RAM)       (chunks)
```

## Micro-Challenges

In [2]:
from pathlib import Path

### MC13.1 : The Safe Open


> Goal : Write to a file without `.close()`.


In [16]:
## from pathlib import Path
path = Path("some_file.txt")
with path.open("w", encoding="utf-8") as f:
    f.write("Hello, World!\n")
    print("File written successfully.")
    ## no need to explicitly close the file  
    ## the context manager does it for us because we used 'with' statement

File written successfully.


In [17]:
## without context manager
my_file = path.open("r", encoding="utf-8")
content = my_file.read()
print("File contents: \n>", content.replace("\n", "\n> "))
my_file.close()  ## MUST explicitly close the file once done

File contents: 
> Hello, World!
> 


> **Deep Dive:** Use `with open(...) as f`. This is a Context Manager. It guarantees file closure even if an exception crashes the block.


---


### MC13.2 : Append vs Write


> Goal : Add a log line to a file without deleting old content.


In [18]:
from pathlib import Path

path = Path("some_file.txt") ## reuse from micro-challenge 13.1

## 'w' wipes the file
with path.open("w", encoding="utf-8") as f:
    f.write("first run [deleted old content]\n")

## 'a' appends to the end
with path.open("a", encoding="utf-8") as f:
    f.write("second run [appended content, preserved first run]\n")

print(path.read_text(encoding="utf-8"))

first run [deleted old content]
second run [appended content, preserved first run]



> **Deep Dive:** Mode `'w'` wipes the file. Mode `'a'` appends to the end. Mode `'x'` fails if file exists (Safety).


---


### MC13.3 : Binary Mode


> Goal : Read an image file.


In [20]:
## from pathlib import Path

path = Path("dd_13_3_image_like.bin")

## Write some bytes (a PNG file starts with this signature)
png_signature = b"\x89PNG\r\n\x1a\n"
fake_png = png_signature + b"not really an image, but bytes are bytes"

with path.open("wb") as f:
    f.write(fake_png)

## Read back as raw bytes
with path.open("rb") as f:
    data = f.read()

print("type:", type(data))
print("starts with PNG signature:", data.startswith(png_signature))
print("first 8 bytes:", data[:8])
print("total bytes read:", len(data))   


type: <class 'bytes'>
starts with PNG signature: True
first 8 bytes: b'\x89PNG\r\n\x1a\n'
total bytes read: 48


> **Deep Dive:** Use mode `'rb'`. Text modes decode bytes to String (UTF-8). Binary modes return raw bytes, essential for images/PDFs.


---


### MC13.4 : Encoding Hell


> Goal : Fix a `"UnicodeDecodeError"`.


In [22]:
## from pathlib import Path

path = Path("dd_13_4_unicode.txt")
text = "Hello — café ☕ 🙂\n"  ## contains non-ascii characters

## Always write with an explicit encoding
path.write_text(text, encoding="utf-8")

## Correct read
print("utf-8 read:", path.read_text(encoding="utf-8").strip())

## Demonstrate why a wrong default can crash (simulate Windows cp1252 mismatch)
raw = path.read_bytes()
try:
    raw.decode("cp1252")  ## this is the problematic default on some Windows setups
    print("cp1252 read succeeded (unexpected):", raw.decode("cp1252").strip())
except UnicodeDecodeError as e:
    print("cp1252 decode failed:", e.__class__.__name__)

utf-8 read: Hello — café ☕ 🙂
cp1252 read succeeded (unexpected): Hello â€” cafÃ© â˜• ðŸ™‚


> **Deep Dive:** Always specify `encoding='utf-8'`. Windows defaults to `'cp1252'`, which crashes on emojis or foreign characters.

[Didn't crash the program, but the string looks corrupted.]


---


### MC13.5 : JSON Serialization


> Goal : Save a dictionary to a file.


In [23]:
import json
## from pathlib import Path

path = Path("dd_13_5_data.json")

data = {
    1: "one",          # int key (allowed in Python)
    2: {"a": 10},
    "ok": True
}

with path.open("w", encoding="utf-8") as f:
    json.dump(data, f, indent=2)

print(path.read_text(encoding="utf-8"))

loaded = json.loads(path.read_text(encoding="utf-8"))
print("loaded keys:", list(loaded.keys()))
print("key types:", {type(k).__name__ for k in loaded.keys()})

{
  "1": "one",
  "2": {
    "a": 10
  },
  "ok": true
}
loaded keys: ['1', '2', 'ok']
key types: {'str'}


> **Deep Dive:** `json.dump()`. JSON is the standard for data exchange.
> **Note:** JSON keys must be strings; Python allows integers, but JSON converts them.


---


### MC13.6 : CSV Parsing


> Goal : Read a CSV into a list of dictionaries.


In [25]:
import csv
## from pathlib import Path

path = Path("dd_13_6_people.csv")

## Note the comma inside the city field (quoted)
csv_text = """
name,city,age
Alice,"London, ON",30
Bob,Toronto,28
""".lstrip()

path.write_text(csv_text, encoding="utf-8")

rows: list[dict[str, str]] = []
with path.open("r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        rows.append(row)

print(rows)
print("Alice --> city:", rows[0]["city"])


[{'name': 'Alice', 'city': 'London, ON', 'age': '30'}, {'name': 'Bob', 'city': 'Toronto', 'age': '28'}]
Alice --> city: London, ON


> **Deep Dive:** Use `csv.DictReader`. It handles quoted strings and delimiters automatically, preventing bugs when data contains commas.


---


### MC13.7 : Buffering


> Goal : Write 1 million lines. Why doesn’t the disk spin 1 million times?


In [33]:
## from pathlib import Path
import time

def write_lines(path: Path, n: int, buffering: int) -> float:
    start = time.perf_counter()
    with path.open("w", encoding="utf-8", buffering=buffering) as f:
        for i in range(n):
            f.write(f"line {i+1:.>10} : " + "x" * 10 + "\n")
    return time.perf_counter() - start

n = 1_000_000  ## 1 million lines
p1 = Path("dd_13_7_buf_1.txt")
p2 = Path("dd_13_7_buf_8192.txt")

t1 = write_lines(p1, n, buffering=1)        # line-buffered-ish
t2 = write_lines(p2, n, buffering=8192)     # chunk buffering

print("buffer=1     seconds:", round(t1, 3))
print("buffer=8192  seconds:", round(t2, 3))
print("speedup:", round(t1 / t2, 2), "x")

buffer=1     seconds: 28.109
buffer=8192  seconds: 5.087
speedup: 5.53 x


> **Deep Dive:** Python (and the OS) uses a **Buffer**. Data accumulates in RAM and is "Flushed" to disk in chunks to save I/O cycles.


---


### MC13.8 : Pathlib


> Goal : Join a folder and filename safely on Windows and Mac.


In [34]:
## from pathlib import Path

folder = Path("data") / "logs" ## creates 'data/logs' path
filename = "app.log"

full_path = folder / filename  ## safe join on any OS
print("full_path:", full_path)
print("as_posix:", full_path.as_posix())

## If you must create dirs:
full_path.parent.mkdir(parents=True, exist_ok=True)
full_path.write_text("hello\n", encoding="utf-8")
print("file exists:", full_path.exists())


full_path: data\logs\app.log
as_posix: data/logs/app.log
file exists: True


> **Deep Dive:** Do not use string concatenation `folder + "/" + file`. Use `pathlib.Path`. It handles OS-specific separators (`\` vs `/`) automatically.


---


### MC13.9 : Custom Context Manager


> Goal : Create a block with `Timer():` that prints time taken.


In [35]:
import time

class Timer:
    def __enter__(self):
        self._start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc, tb):
        elapsed = time.perf_counter() - self._start
        print(f"time taken: {elapsed:.6f}s")
        # return False means: don't suppress exceptions
        return False


with Timer():
    total = 0
    for i in range(1_000_00):
        total += i
print("total:", total)

try:
    with Timer():
        raise ValueError("example error")
except ValueError:
    print("exception observed outside the block")


time taken: 0.060073s
total: 4999950000
time taken: 0.000006s
exception observed outside the block


> **Deep Dive:** Implement `__enter__` (start timer) and `__exit__` (end timer).


---


### MC13.10 : Pickle (The Warning)


> Goal : Save a Python Object (Class) to file.


In [36]:
import pickle
## from pathlib import Path

class User:
    def __init__(self, name: str, points: int):
        self.name = name
        self.points = points

    def __repr__(self) -> str:
        return f"User(name={self.name!r}, points={self.points})"


path = Path("dd_13_10_user.pkl")

u1 = User("Saarah", 42)

with path.open("wb") as f:
    pickle.dump(u1, f)

with path.open("rb") as f:
    u2 = pickle.load(f)

print("u1:", u1)
print("u2:", u2)
print("same object identity:", u1 is u2)
print("same value-like state:", (u1.name, u1.points) == (u2.name, u2.points))
print("types:", type(u1), type(u2))


u1: User(name='Saarah', points=42)
u2: User(name='Saarah', points=42)
same object identity: False
same value-like state: True
types: <class '__main__.User'> <class '__main__.User'>


> **Deep Dive:** Use `pickle`.
> **Warning:** Never unpickle data from untrusted sources. It can execute arbitrary code (Security Risk).


---
